In [ ]:
from rdflib import Graph

g_gemet = Graph()
g_gemet.parse("/home/d-ladesteijn/p2f/p2f-api/p2f_api/adminutils/skos/gemet.rdf", format="xml")

print(f"Number of rows: {len(g_gemet)}")
for text_gemet in g_gemet:
    print(text_gemet)

In [ ]:
import pandas as pd

gemet = pd.DataFrame(
    [(str(subject), str(predicate), str(obj)) for subject, predicate, obj in g_gemet],
    columns=["subject","predicate", "object"]
)

print(gemet)
gemet.to_parquet("gemet_full_data.parquet", index=False)

In [ ]:
import pandas as pd

rows = []

for subject, predicate, obj in g_gemet:
    rows.append({
        "subject": str(subject),
        "object": str(obj),
        "object_type": type(obj).__name__,
        "object_lang": getattr(obj, "language", None),
    })

df_gemet = pd.DataFrame(rows)
print(df_gemet)

In [ ]:
from rdflib import Literal

languages = sorted({
    obj.language
    for _, _, obj in g_gemet
    if isinstance(obj, Literal) and obj.language
})

print(languages)

In [ ]:
filtered_df_gemet = df_gemet.loc[
    df_gemet["subject"].str.startswith("http://www.eionet.europa.eu", na=False)
    & df_gemet["object_type"].eq("Literal")
    & df_gemet["object_lang"].isin(["en", "en-US"])
].copy()

print(filtered_df_gemet)

filtered_df_gemet.to_parquet("gemet_filtered_data.parquet", index=False)